# LLM graph iteration demo

Anton Antonov   
January 2026

---

## Introduction

This notebook shows how to do iterative refinements of an LLM-based (AI) agent results using a iterative evaluation of an (acyclic) LLM-graph.

---

## Setup

In [58]:
use LLM::Graph;
use WWW::Ollama;

In [59]:
sink my $conf-ollama = llm-configuration('ollama', model => 'gemma3:1b')

In [60]:
sink my $conf41 = llm-configuration('ChatGPT', model => 'gpt-4.1', temperature => 0.55, max-tokens => 8192);

In [61]:
sink my $conf = $conf-ollama;

----

## Graph nodes

The code below implements the following workflow:

1. Generate the first draft

2. Determine whether the draft should criticized 

    - LLM can be used, or (deterministic) lookup of a verdict word or phrase.

3. If, yes, critique the draft

    - The critic gives a verdict "BAD", "ACCEPTABLE", and "GOOD" _and_ criticism notes.

4. If the verdict is not "GOOD" generate a new revision using criticism notes

5. The iterations stops when the critique gives the verdict "GOOD" or max iterations is reached.

### Generation rules

In [62]:
sink my %generation-rules =
    generate => sub ($topic) { "Generate a 200 words thesis text for the topic $topic." ~ llm-prompt('NothingElse')('thesis text without comments')} 
;

### Critique LLM-function

In [63]:
my &cFunc = llm-function( -> $text {
    [
        "Write a VERY SHORT critique of the thesis below.",
        "Give your verdict to be one of 'BAD', 'ACCEPTABLE', 'GOOD'.",
        "If you verdict is 'BAD' or 'ACCEPTABLE' give improvement directions only.",
        "THESIS:\n", 
        $text ~~ Str:D ?? $text !! $text<revision>,
        "THESIS-END\n",
        'Give critique as a JSON dictionary of the form: { "verdict": ..., "comments" : [ ...] }',
        llm-prompt('NothingElse')('JSON')
    ].join("\n")
}, e => $conf, form => sub-parser('JSON'):drop);

LLM::Function(-> **@args, *%args { #`(Block|5053401571496) ... }, 'ollama')

Critique prompt variant:

```raku
    [
        "Write a VERY SHORT critique of the thesis below.",
        "Evaluate with questions like:",
        "1) Is it too short? 2) Does it have at least two sections? 3) Is it easy to read?",
        "Give your verdict to be one of 'BAD', 'ACCEPTABLE', 'GOOD'.\n",
        "THESIS:\n", 
    ]
```    

### Revision LLM-function

In [64]:
my &rFunc = llm-function(-> $critique-form {
    [
        "Here is a thesis:",
        "\nTHESIS:\n", 
        $critique-form<revision>,
        "\nTHESIS-END\n",
        "And here is a critique of it:",
        "CRITIQUE:\n",
        $critique-form<critique>,
        "END-CRITIQUE\n",
        "Improve the thesis text based on the critique.",
        "Do not comment, annotate, or give options"
    ].join("\n")
}, e => $conf);

LLM::Function(-> **@args, *%args { #`(Block|5053763552400) ... }, 'ollama')

### Refinement rules

In [65]:
my %empty-form = critique => '', critique-iteration => 0, revision => '', revision-iteration => 0;

sink my %revision-rules =

    decide => { 
        eval-function => sub ($text) { 
            $text ~~ Str:D || !($text ~~ Map:D && $text<critique>.join("\n").contains('GOOD'))
        } 
    },

    critique => {
        llm-function => sub ($text) { &cFunc($text) },
        
        test-function => sub ($decide) { $decide.raku.lc.contains('true') }
    },

    critique-form => {
        eval-function => sub ($text, $critique) {
            my $form = $text ~~ Map:D ?? $text.clone !! merge-hash(%empty-form , {revision => $text});
            $form<critique> = $critique;
            $form<critique-iteration> += 1;
            return $form;
        },
        
        test-function => sub ($critique) { $critique.defined }
    },

    revise => {
        llm-function => sub ($critique-form) { &rFunc($critique-form) },
        
        test-function => sub ($critique-form) { $critique-form.defined },
    },

    revise-form => {
        eval-function => sub ($critique-form, $revise) {
            my %form = $critique-form.clone; 
            %form<revision> = $revise;
            %form<revision-iteration> += 1;
            return %form;
        },
        
        test-function => sub ($revise) { $revise.defined }
    },

    finalize => {
        eval-function => sub ($text, $revise-form) { $revise-form.defined ?? $revise-form !! $text}
    }  
;

---


## Build the graphs

In [66]:
my $g1 = LLM::Graph.new(%generation-rules, e => $conf):async;
my $g2 = LLM::Graph.new(%revision-rules, e => $conf):async;

(:$g1, :$g2)

(g1 => LLM::Graph(size => 1, nodes => generate) g2 => LLM::Graph(size => 6, nodes => critique, critique-form, decide, finalize, revise, revise-form))

---

## Run and inspect state transitions

In [67]:
$g1.eval({ topic => "why cyclic graphs help with iterative writing" })

LLM::Graph(size => 1, nodes => generate)

In [68]:
my $text = $g1.nodes<generate><result>

The incorporation of cyclic graphs within a writing workflow significantly supports iterative development and improved clarity. These graphical representations offer a valuable framework for visualizing and dissecting narrative elements, allowing writers to systematically reassess character motivations, plot progression, and worldbuilding concepts. Utilizing a cyclic approach encourages iterative experimentation – revisiting sections, modifying elements, and tracing changes across a structured diagram. This dynamic process, facilitated by the graph’s inherent connectivity, promotes a better understanding of potential narrative tangents and unintended consequences. Furthermore, the cyclical nature makes it significantly more intuitive to pinpoint key narrative areas, encouraging focused writing while maintaining a continuous developmental process. 

| Section         | Point of View  |  Revision Request | Flow Direction | Potential Issue          |
| :--------------- | :------------- | 

In [69]:
$text.words.elems

197

In [70]:
#% html
&cFunc($text)
==> to-html

verdict GOOD comments Strengthen theme introduction - introduce central conflict immediately
, Increase sensory detail to enhance visual representation
, Ensure progression - create a deliberate, branching narrative pathway.

In [71]:
$g2.eval({ :$text })

LLM::Graph(size => 6, nodes => critique, critique-form, decide, finalize, revise, revise-form)

In [72]:
#% markdown
$g2.nodes<finalize><result>

{critique => [verdict GOOD comments Expand on the 'tangents' and 'unintended consequences' described - provide specific examples. Replace some repetitive phrasing (e.g., 'dynamic process') with more concise wording. Consider adding a brief explanation of *how* the cyclic graph visually aids understanding. Strengthen the opening by clearly state the *thesis's core promise*.], critique-iteration => 1, revision => Okay, here’s a revised version incorporating your feedback, and designed to be more compelling:

**THESIS:**

The incorporation of cyclic graphs within a writing workflow exponentially strengthens iterative development and substantially improves clarity. These graphical representations offer a powerful framework for visually dissecting narrative elements – particularly character motivations, plot progression, and worldbuilding – allowing writers to systematically reassess and rigorously explore potential deviations. Utilizing this cyclical approach encourages iterative experimentation: revisiting specific sections, modifying elements, and tracking changes across a structured diagram.  The inherent connectivity of the graph fosters intuitive pinpointing of critical narrative areas, promoting focused work and maintaining a continual developmental process, ultimately leading to enhanced narrative understanding and coherence.

| Section         | Point of View       | Revision Request        | Flow Direction | Potential Issue            |
| :--------------- | :--------------- | :------------------------- | :-------------- | :-------------------------- |
| Introduction    | Narrative       | Explain Theme Focus   | Initial      | Overly vague – needs to suggest *why* narrative clarification is vital |
| Developing    | Character        | Detail Depth of Motivation| Exploration   | Lacks tangible illustrations – needs concrete examples of what “deep” motivation looks like. |
| Staging        | Setting           | Sensorimotor Details      | Visualization   | Reduced sensory detail could obscure nuances.            |
| Climax        | Conflict          | Character Reaction        | Backward  | “Sudden escalation” is a bit vague; provide an illustration of how this might manifest in the graph.|
| Resolution     | Outcome           | Impact on Characters    | Forward      |  The focus isn't explicitly about *impact*; it's simply about forward movement, potentially missing the significance of that motion.|

**Rationale for Changes:**

*   **Stronger Opening:** Phrases like "exponentially strengthens" and "powerful framework" immediately anchor the idea and signal what we're going to cover.
*   **Specific Examples:** Replaced generic phrases with things like "particularly character motivations" and added "exploring potential deviations." Suggestion of examples like showing *how* character motivation is explored increases buy-in.
*   **More Concise Wording:**  “Remarkably facilitates understanding” became “enforces intuitive understanding.”  “Visual Clutter” changed to "reduced sensory detail."
*   **Improved Flow:** Transition phrases have been refined to provide a better logical progression.
*   **Focus on the Core Promise:** Reinforced the thesis's core benefit - improved narrative coherence.


Let me know if you would like any further refinements or revisions., revision-iteration => 1}

In [73]:
$g1.clear;
$g2.clear;

$g1.eval({ topic => "Why cyclic graphs help with iterative writing?" });
my $text = $g1.nodes<generate><result>;

for (^4) -> $iter {
  say '=' x 10, $iter, '=' x 10;
  
  $g2.eval({:$text});  
  
  my $revision = $g2.nodes<finalize><result>;
  last if $revision eq $text;
  $text = $revision;

  say 'critique => ', $g2.nodes<critique><result>;
  say "-=" x 10, "-";
  say to-json($text);
}

==========0==========
critique => [verdict => ACCEPTABLE comments => [More concrete examples of pattern recognition would strengthen the thesis. Reference specific cognitive benefits. Emphasize the *feeling* of creation rather than just structure. Discuss narrative flow with greater detail.]]
-=-=-=-=-=-=-=-=-=-=-
{
  "critique-iteration": 1,
  "critique": [
    {
      "verdict": "ACCEPTABLE"
    },
    {
      "comments": [
        "More concrete examples of pattern recognition would strengthen the thesis. Reference specific cognitive benefits.",
        "Emphasize the *feeling* of creation rather than just structure. Discuss narrative flow with greater detail."
      ]
    }
  ],
  "revision": "Okay, here’s an enhanced thesis, building on the critique and incorporating more details:\n\n**THESIS:**\n\nCyclic graphs effectively guide iterative writing due to their inherently structured patterns of task repetition. This repeated framework stimulates a natural understanding of structure

### Final output

In [74]:
#% markdown
$text<revision>

Okay, here’s an improved thesis, building on your critique and focusing on enhanced reasoning and a stronger narrative flow:

**Improved Thesis:**

Cyclic graph construction cultivates narrative productivity through the inherent, repeated restructuring of tasks – mirroring a pattern of gradual refinement.  This structured process significantly diminishes creative stalling, fostering an intuitive understanding of structural nuances crucial for effective storytelling. Coupled with the iterative process of visual mapping and the seamless convergence of elements, writers discover a pathway for expansive and focused narrative construction, leading to efficiently produced text, moving beyond initial brainstorming and into polished storytelling.

**Explanation of Changes & Reasoning (Reinforcing the Critique):**

*   **Stronger Opening:** I rephrased the opening to immediately communicate the core benefit – productivity.
*   **Focus on the Core Mechanism:** By framing it as “inherent, repeated restructuring of tasks”, we emphasize the foundational mechanism—that the cycles are already in place. This anchors the concept.
*   **Highlighting the "Why" – Stalled Creativity:** Refining the mention of "creative stalling" to "visual mapping and the seamless convergence of elements" links it directly to the *process* – it reveals the *method* of overcoming that stagnation.
*  **Elaborated on Narrative Flow:** Changed “effortless text construction” to “efficiently produced text” – this adds a tangible outcome and acknowledges quality production.
* **Refined Conclusion**: The last sentence, "moving beyond initial brainstorming and into polished storytelling", establishes a progression from the initial creative act to the final product better conveying the intention.

The revision aims for a greater sense of dynamism - suggesting a continuous creative flow enhanced by the tool of cycles.

In [75]:
#% markdown
$g2.nodes<critique-form>

{eval-function => sub { }, input => [text critique], result => (Any), spec-type => (Callable), test-function => sub { }, test-function-input => [critique], test-function-result => False}

----

## LLM graph plots

In [76]:
#% html
my %opts = engine => 'dot', :9graph-size, node-width => 1.2, theme => 'default';
[
    generation-graph => $g1.dot(|%opts, graph-size => 1.5, :svg),
    revision-graph => $g2.dot(|%opts, :5graph-size, :svg)
]
==> to-html-table()

generation-graph revision-graph <?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
 
 generate_Cluster 
 
 
<!-- generate -->
 
 generate 
 
 generate 
 
<!-- topic -->
 
 topic 
 
 topic 
 
<!-- topic->generate -->
 
 topic->generate 
 
 
 
 
 
 <?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- critique -->
 
 critique 
 
 critique 
 
<!-- critique-form -->
 
 critique-form 
 
 critique-form 
 
<!-- critique->critique-form -->
 
 critique->critique-form 
 
 
 
<!-- revise -->
 
 revise 
 
 revise 
 
<!-- critique-form->revise -->
 
 critique-form->revise 
 
 
 
<!-- revise-form -->
 
 revise-form 
 
 revise-form 
 
<!-- critique-form->revise-form -->
 
 critique-form->revise-form 
 
 
 
<!-- decide -->
 
 decide 
 
 decide 
 
<!-- decide->critique -->
 
 decide->critique 
 
 
 
<!-- finalize -->
 
 finalize 
 
 finalize 
 
<!-- revise->revise-form -->
 
 revise->revise-form 
 
 
 
<!-- revise-form->finalize -->
 
 revise-form->finalize 
 
 
 
<!-- text -->
 
 text 
 
 text 
 
<!-- text->critique -->
 
 text->critique 
 
 
 
<!-- text->critique-form -->
 
 text->critique-form 
 
 
 
<!-- text->decide -->
 
 text->decide 
 
 
 
<!-- text->finalize -->
 
 text->finalize